In [1]:
# install the required libaries

! pip install accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.30.2 langchain chromadb google-generativeai --quiet
! pip install pypdf --quiet # required for LangChain pdf reader
! pip install tiktoken --quiet # required for splitting document by number of tokens
! pip install sentence_transformers --quiet # for creating embeddings

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cudf 23.6.1 requires cupy-cuda11x>=12.0.0, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.6 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 11.0.0 which is incompatible.
cmudict 1.0.13 requires importlib-metadata<6.0.0,>=5.1.0, but you have importlib-metadata 6.0.1 which is incompatible.
dask-cuda 23.6.0 requires dask==2023.3.2, but you have dask 2023.7.0 which is incompatible.
dask-cudf 23.6.1 requires dask==2023.3.2, but you have dask 2023.7.0 which is incompatible.
distributed 2023.3.2.1 requires dask==2023.3.2, but you have dask 2023.7.0 which is incompatible.
google-cloud-pubsublite 1.8.2 requires overrides<7.0.0,>=6.0.1, but you have overrides 7.4.0 which is incompatible.
jupyterlab-lsp 4.2.0 requires jupyte

In [2]:
llama2_paper_path = '/kaggle/input/nlp-and-llm-related-arxiv-papers/LLaMA- Open and Efficient Foundation Language Models.pdf'

### **Loading the document**

In [3]:
# import the LangChain pdf document loader
from langchain.document_loaders import PyPDFLoader

In [4]:
# Load and create pages
loader = PyPDFLoader(file_path=llama2_paper_path)
pages = loader.load_and_split()

### **Creating text chunks for Document**

In [5]:
from langchain.text_splitter import CharacterTextSplitter

loader = PyPDFLoader(file_path=llama2_paper_path)
documents = loader.load()

# we split the data into chunks of 1,000 characters, with an overlap
# of 200 characters between the chunks, which helps to give better results
# and contain the context of the information between chunks

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(documents)

# Total number of text chunks
print(len(documents))

# length of a single document
print(len(documents[0].page_content))

27
4056


### **Creation of Embeddings**

We will use the open source sentence transformer embedding to create the embedding , you can also use the OpenAI embedding to create the embeddings.

In [6]:
from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

### **Vector Store**

In [7]:
from langchain.vectorstores import Chroma

# load embeddings into Chroma - need to pass docs , embedding function and path of the db

db = Chroma.from_documents(documents,
                           embedding=embeddings,
                           persist_directory='./llama-db')

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
# saving into disk for future use
db.persist()

### **Creating a MultiQueryRetriever using LLM (llama-2)**

In [9]:
import torch 
import transformers # HF import
from langchain import HuggingFacePipeline # To build the HF pipeline using Llama-2
from langchain import PromptTemplate,  LLMChain # To create PromptTemplate and LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM , AutoModel  # For creating the model and tokenizer

In [10]:
%%time

model_name = 'daryl149/llama-2-7b-chat-hf' # Model path for Llama-2 finetuned chat model

# tokenizer creation
tokenizer = AutoTokenizer.from_pretrained(model_name)

# importing the pretrained model
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             device_map='auto',
                                             torch_dtype=torch.float16,
                                            load_in_4bit=True
                                             )

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

CPU times: user 18 s, sys: 29.3 s, total: 47.3 s
Wall time: 1min 40s


In [11]:
# creating a hugging face pipeline 
from transformers import pipeline


pipe = pipeline("text-generation",
                model=model,
                tokenizer= tokenizer,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                max_new_tokens = 256,
                do_sample=True,
                top_k=1,
                num_return_sequences=1,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.2
                )

Xformers is not installed correctly. If you want to use memory_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.


In [12]:
llm=HuggingFacePipeline(pipeline=pipe, model_kwargs={'temperature':0})

In [13]:
from langchain.globals import set_verbose

set_verbose(True)

from langchain.globals import set_debug

set_debug(True)

# Set logging for the queries
import logging
logging.basicConfig()
logging.getLogger('langchain.retrievers.multi_query').setLevel(logging.INFO)

MultiQueryRetriever

In [14]:
from langchain.retrievers.multi_query import MultiQueryRetriever
mq_retriever = MultiQueryRetriever.from_llm(retriever=db.as_retriever(),llm=llm)

In [15]:
query = "what is so special about llama 2?"

output = mq_retriever.get_relevant_documents(query=query)

[chain/start] [1:retriever:Retriever > 2:chain:LLMChain] Entering Chain run with input:
{
  "question": "what is so special about llama 2?"
}
[llm/start] [1:retriever:Retriever > 2:chain:LLMChain > 3:llm:HuggingFacePipeline] Entering LLM run with input:
{
  "prompts": [
    "You are an AI language model assistant. Your task is \n    to generate 3 different versions of the given user \n    question to retrieve relevant documents from a vector  database. \n    By generating multiple perspectives on the user question, \n    your goal is to help the user overcome some of the limitations \n    of distance-based similarity search. Provide these alternative \n    questions separated by newlines. Original question: what is so special about llama 2?"
  ]
}
[llm/end] [1:retriever:Retriever > 2:chain:LLMChain > 3:llm:HuggingFacePipeline] [4.59s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "  \n1. What are the key features that make llama 2 stand out in its fiel

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
print(output)

[Document(page_content='LLaMA GPT3 OPT\nGender 70.6 62.6 65.7\nReligion 79.0 73.3 68.6\nRace/Color 57.0 64.7 68.6\nSexual orientation 81.0 76.2 78.6\nAge 70.1 64.4 67.8\nNationality 64.2 61.6 62.9\nDisability 66.7 76.7 76.7\nPhysical appearance 77.8 74.6 76.2\nSocioeconomic status 71.5 73.8 76.2\nAverage 66.6 67.2 69.5\nTable 12: CrowS-Pairs. We compare the level of bi-\nases contained in LLaMA-65B with OPT-175B and\nGPT3-175B. Higher score indicates higher bias.\n5.2 CrowS-Pairs\nWe evaluate the biases in our model on the CrowS-\nPairs (Nangia et al., 2020). This dataset allows to\nmeasure biases in 9 categories: gender, religion,\nrace/color, sexual orientation, age, nationality, dis-\nability, physical appearance and socioeconomic sta-\ntus. Each example is composed of a stereotype and\nan anti-stereotype, we measure the model prefer-\nence for the stereotypical sentence using the per-\nplexity of both sentences in a zero-shot setting.\nHigher scores thus indicate higher bias. We co